# Figuras do ensaio sobre emissões e fornecedores na MIP 2015

Este notebook lê exclusivamente os resultados publicados da investigação. A sequência é concentração, fornecedores, associação ambiental–produtiva e ajuste descritivo por tamanho. Nenhuma nova centralidade é estimada. As emissões são um **cenário exploratório com base ambiental categoria C**, não um inventário validado de 2015.

Execute da raiz do repositório ou desta pasta. As quatro figuras são gravadas aqui em PNG, SVG e PDF; os outputs históricos são somente lidos. O [ensaio](../../../docs/ensaio_redes_emissoes_2015.md) e o [roteiro da história](../../../docs/historia_ensaio_redes_2015.md) usam essas mesmas figuras.

In [1]:
from pathlib import Path
import hashlib, json, platform
import numpy as np
import pandas as pd
import scipy
from scipy.stats import spearmanr
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D

raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'raw/manifesto.csv').exists())
base = raiz/'outputs/investigacao_redes_2015'
pasta = base/'ensaio'
pasta.mkdir(exist_ok=True)
arquivos = ['entradas_alinhadas.csv', 'metricas_setoriais.csv', 'matriz_L.csv', 'concentracao.csv', 'cenarios_coeficientes.csv']
hashes = {f: hashlib.sha256((base/f).read_bytes()).hexdigest() for f in arquivos}
t = pd.read_csv(base/'entradas_alinhadas.csv', dtype={'atividade': str}).set_index('atividade')
m = pd.read_csv(base/'metricas_setoriais.csv', dtype={'atividade': str}).set_index('atividade').copy()
assert t.index.equals(m.index) and t.index.is_unique and len(t) == 67
L = pd.read_csv(base/'matriz_L.csv', index_col=0, dtype=str)
assert list(L.index) == list(t.index) == list(L.columns)
L = L.astype(float).to_numpy()
c = L.sum(axis=1) - np.diag(L)
e, x = t.e.to_numpy(), t.x.to_numpy()
gamma = (3*t.gamma2011.to_numpy() + 4*t.gamma2018.to_numpy())/7
np.testing.assert_allclose(e, gamma*x, atol=1e-8)
np.testing.assert_allclose(c, m.L_saida, atol=1e-12)
np.testing.assert_allclose(e, m.e, atol=1e-8)
m['rank_L'] = pd.Series(c, index=t.index).rank(ascending=False, method='average')
m['rank_S'] = m.S_media.rank(ascending=False, method='average')
m['rank_A'] = m.A_saida.rank(ascending=False, method='average')
rho = spearmanr(e,c).statistic
postos = pd.DataFrame({'emissao':e, 'centralidade':c, 'producao':x}, index=t.index).rank(method='average')
X = np.column_stack([np.ones(len(t)), postos.producao])
# A parcial é a correlação dos resíduos dos postos, sem ranqueá-los novamente.
residuos = postos[['emissao','centralidade']] - X @ np.linalg.lstsq(X, postos[['emissao','centralidade']], rcond=None)[0]
parcial = residuos.corr().iloc[0,1]
cen = pd.read_csv(base/'cenarios_coeficientes.csv').set_index('cenario')
np.testing.assert_allclose([rho,parcial], cen.loc['2015_interpolado',['rho_e_Lsaida','parcial_x_Lsaida']].to_numpy(dtype=float), atol=1e-12)

AZUL, LARANJA, CINZA, TEXTO = '#176A80', '#C66A30', '#A9B3BA', '#26343D'
cores = {'3500':'#C66A30', '3680':'#7753A0', '4180':'#AD4764', '4580':'#176A80', '1991':'#66833A', '4900':'#337AB7'}
rotulos = {'3500':'Energia/utilidades', '3680':'Água/resíduos', '4180':'Construção', '4580':'Comércio', '1991':'Refino', '4900':'Transporte terrestre'}
plt.rcParams.update({'font.family':'DejaVu Sans', 'font.size':11, 'axes.titlesize':13,
    'axes.labelsize':11, 'text.color':TEXTO, 'axes.labelcolor':TEXTO,
    'xtick.color':TEXTO, 'ytick.color':TEXTO, 'axes.spines.top':False,
    'axes.spines.right':False, 'axes.edgecolor':'#BBC3C8', 'figure.facecolor':'white',
    'axes.facecolor':'white', 'savefig.facecolor':'white', 'svg.fonttype':'none', 'pdf.fonttype':42})
pt = FuncFormatter(lambda value,pos: f'{value:g}'.replace('.',','))

def acabamento(fig, numero, titulo, subtitulo, nota):
    fig.text(.075,.958,f'FIGURA {numero}   •   MIP BRASILEIRA 2015',fontsize=10,color=AZUL,weight='bold')
    fig.text(.075,.910,titulo,fontsize=20,weight='bold')
    fig.text(.075,.865,subtitulo,fontsize=10.5,color='#52616B')
    fig.text(.075,.045,nota,fontsize=9,color='#52616B',linespacing=1.5)

def salvar(fig,nome):
    for ext in ['png','svg','pdf']:
        fig.savefig(pasta/f'{nome}.{ext}',dpi=190)
    plt.close(fig)
    print('Figura gravada:',nome,'(PNG, SVG, PDF)')

print('67 atividades; rho =',rho,'; parcial =',parcial)
print('Versões:',platform.python_version(),np.__version__,pd.__version__,scipy.__version__,matplotlib.__version__)
print('SHA-256 das entradas:',json.dumps(hashes,ensure_ascii=False,indent=2))


67 atividades; rho = 0.38279192273924506 ; parcial = 0.1582122646440335
Versões: 3.12.13 2.5.3 3.0.5 1.17.1 3.11.1
SHA-256 das entradas: {
  "entradas_alinhadas.csv": "a34d5de00a5bc2d483275d8ffb5464dd4dfb7f82e478e87816c4a199d161211e",
  "metricas_setoriais.csv": "7f28cfab8008845deca522f0b1ffd30fff80854fbb5f08890c9cbdd0ce61a87f",
  "matriz_L.csv": "a4a116796378ae6a077473cd0661db7723b49f11e2db599bd41e72766304f2f3",
  "concentracao.csv": "5eadc44aa47d03ec90a2b3d13157f6c68597f1ae3f9d974fd94fd6dcf3ff0f69",
  "cenarios_coeficientes.csv": "0a109a38229155b860434af76eb402ce9022f851c13aa9427db5e403a50c1f43"
}


## 1 Concentração das emissões e da produção

Curvas de Lorenz ordenam cada variável separadamente; não dizem se os maiores setores são os mesmos. Gini é calculado por diferenças absolutas; top 10 usa as dez maiores observações de cada variável. As conclusões se restringem ao cenário interpolado.

In [2]:
fig = plt.figure(figsize=(11.6,7.5))
ax = fig.add_axes([.10,.19,.52,.61])
acumulado = np.linspace(0,100,len(t)+1)
concentracao = {}
for nome,valores,cor,estilo in [('Emissões do cenário',e,LARANJA,'-'),('Produção bruta',x,AZUL,'--')]:
    lorenz = np.r_[0,np.cumsum(np.sort(valores))/valores.sum()]*100
    ax.plot(acumulado,lorenz,label=nome,color=cor,lw=2.6,ls=estilo)
    concentracao[nome] = {'top10':100*np.sort(valores)[-10:].sum()/valores.sum(),
        'gini':np.abs(valores[:,None]-valores[None,:]).sum()/(2*len(valores)*valores.sum())}
ax.plot([0,100],[0,100],color='#BFC8CD',lw=1,ls=':',label='Distribuição igualitária')
ax.set(xlim=(0,100),ylim=(0,100),xlabel='Atividades acumuladas (%)',ylabel='Valor acumulado (%)')
ax.set_aspect('equal')
ax.grid(alpha=.16)
ax.legend(loc='upper left',frameon=False,fontsize=10)
fig.text(.68,.735,'DEZ MAIORES',fontsize=10,color='#52616B',weight='bold')
fig.text(.68,.672,'48,3%',fontsize=30,weight='bold',color=LARANJA)
fig.text(.68,.635,'das emissões do cenário',fontsize=11)
fig.text(.68,.562,'50,4%',fontsize=30,weight='bold',color=AZUL)
fig.text(.68,.525,'da produção bruta',fontsize=11)
fig.text(.68,.405,'GINI',fontsize=10,color='#52616B',weight='bold')
fig.text(.68,.365,'Emissões   0,504\nProdução   0,534',fontsize=12,linespacing=1.8,va='top')
ac = pd.read_csv(base/'concentracao.csv',index_col=0)
np.testing.assert_allclose([concentracao['Emissões do cenário']['gini'],concentracao['Produção bruta']['gini']],ac.loc[['Emissões próprias e','Produção bruta x'],'gini'])
acabamento(fig,1,'A concentração ambiental acompanha a econômica',
    'Curvas de Lorenz • 67 atividades ordenadas separadamente em cada variável',
    'Fonte: outputs da investigação, MIP/IBGE e coeficientes Sanguinet–Azzoni. Emissões: cenário exploratório.\nA comparação não identifica os mesmos setores no topo, nem demonstra dominância em todos os indicadores.')
salvar(fig,'figura_1_concentracao')


Figura gravada: figura_1_concentracao (PNG, SVG, PDF)


![Curvas de Lorenz de emissão e produção](figura_1_concentracao.png)

## 2 Fornecedores centrais

Barras mostram soma de L por linha, excluindo a diagonal. Postos de S e A são calculados sobre as 67 atividades. Comparar postos não equivale a comparar as magnitudes das três medidas.

In [3]:
top = m.nlargest(10,'L_saida')
nomes_curtos = ['Comércio','Refino de petróleo','Transporte terrestre','Intermediação financeira','Agricultura',
    'Jurídicas, contábeis e consultoria','Energia e utilidades','Químicos orgânicos e inorgânicos',
    'Serviços administrativos','Extração de petróleo e gás']
assert top.index.tolist() == ['4580','1991','4900','6480','0191','6980','3500','2091','7880','0680']
assert set(top.index) == set(m.nlargest(10,'S_media').index)
fig = plt.figure(figsize=(12.8,7.7))
ax = fig.add_axes([.28,.20,.48,.59])
ypos = np.arange(10)
ax.barh(ypos,top.L_saida,color=AZUL,height=.64)
ax.set_yticks(ypos,nomes_curtos,fontsize=10.5)
ax.invert_yaxis()
ax.set_xlim(0,6.8)
ax.set_xlabel('Centralidade como fornecedor • soma de L fora da diagonal')
ax.xaxis.set_major_formatter(pt)
ax.xaxis.grid(True,alpha=.17)
ax.set_axisbelow(True)
ax.tick_params(axis='y',length=0)
for j,(sid,row) in enumerate(top.iterrows()):
    ax.text(row.L_saida+.08,j,f'{row.L_saida:.2f}'.replace('.',','),va='center',fontsize=10)
table = fig.add_axes([.80,.20,.15,.59],sharey=ax)
table.set_xlim(0,2)
table.axis('off')
for j,(sid,row) in enumerate(top.iterrows()):
    table.text(.45,j,f'{row.rank_S:.0f}',ha='center',va='center',fontsize=12)
    table.text(1.5,j,f'{row.rank_A:.0f}',ha='center',va='center',fontsize=12)
fig.text(.833,.81,'S',ha='center',weight='bold',fontsize=11)
fig.text(.912,.81,'A',ha='center',weight='bold',fontsize=11)
fig.text(.87,.835,'POSTOS',ha='center',fontsize=10,color='#52616B')
acabamento(fig,2,'Os principais fornecedores mudam pouco entre A, L e S',
    'Dez primeiros em L • S normaliza as cadeias; A considera vínculos diretos • 1 = mais central',
    'Fonte: MIP/IBGE, cálculos da investigação. ρ(L,S) = 0,998; top 10 coincidentes; A compartilha nove setores.\nL, S e A são transformações da mesma MIP. A figura não usa coeficientes de emissão.')
salvar(fig,'figura_2_fornecedores')


Figura gravada: figura_2_fornecedores (PNG, SVG, PDF)


![Dez fornecedores mais centrais em L e postos em S e A](figura_2_fornecedores.png)

## 3 Emissão e posição como fornecedor

Todos os 67 setores são exibidos em escala linear, com pontos de mesmo tamanho. Os seis contrastes destacados são os já discutidos na investigação; não constituem grupos naturais. Não se aplica limiar, índice composto ou fronteira de Pareto.

In [4]:
fig = plt.figure(figsize=(11.6,7.7))
ax = fig.add_axes([.10,.19,.84,.61])
share = 100*e/e.sum()
ax.scatter(share,c,s=35,color=CINZA,alpha=.75,edgecolors='white',linewidth=.6,zorder=2)
offsets = {'4580':(14,4),'1991':(-10,15),'4900':(18,-13),'3500':(-13,12),'3680':(-12,-22),'4180':(-12,15)}
for sid,label in rotulos.items():
    i = t.index.get_loc(sid)
    ax.scatter(share[i],c[i],s=70,color=cores[sid],edgecolors='white',linewidth=.9,zorder=4)
    dx,dy = offsets[sid]
    ax.annotate(label,(share[i],c[i]),xytext=(dx,dy),textcoords='offset points',
        ha='left' if dx>0 else 'right',va='center',fontsize=10.5,color=cores[sid],weight='bold',
        bbox=dict(facecolor='white',edgecolor='none',alpha=.88,pad=1.7),
        arrowprops=dict(arrowstyle='-',color=cores[sid],lw=.65),zorder=5)
ax.text(.025,.95,'Spearman ρ = 0,383\nn = 67 atividades',transform=ax.transAxes,ha='left',va='top',fontsize=12,
    bbox=dict(facecolor='white',edgecolor='#DFE5E8',pad=9))
ax.set(xlim=(-.15,9.0),ylim=(-.22,6.55),xlabel='Participação nas emissões do cenário (%)',
    ylabel='Centralidade como fornecedor • soma de L fora da diagonal')
ax.xaxis.set_major_formatter(pt)
ax.yaxis.set_major_formatter(pt)
ax.grid(alpha=.14,zorder=0)
acabamento(fig,3,'Emissão e centralidade produzem ordenações distintas',
    'Associação positiva com contrastes setoriais • mesma área para todos os pontos',
    'Fonte: outputs da investigação. Emissões: cenário exploratório com preços e alocação ambiental não validados.\nAs posições de Água/resíduos e Transporte exigem especial cautela; o eixo vertical não mede essencialidade.')
salvar(fig,'figura_3_emissao_centralidade')


Figura gravada: figura_3_emissao_centralidade (PNG, SVG, PDF)


![Dispersão de emissão e centralidade em L](figura_3_emissao_centralidade.png)

## 4 Associação antes e depois do ajuste por tamanho

Usam-se postos crescentes: 67 indica o maior valor, ao contrário do ranking de fornecedores da figura 2. Projetam-se os postos de emissão e centralidade sobre constante e posto de x. No segundo painel, os resíduos não são ranqueados novamente. Sua correlação de Pearson é a parcial de postos. O ajuste é descritivo e não identifica um efeito causal do tamanho.

In [5]:
fig,axes = plt.subplots(1,2,figsize=(12.8,7.3))
fig.subplots_adjust(left=.08,right=.96,bottom=.23,top=.76,wspace=.28)
for ax,data in zip(axes,[postos,residuos]):
    ax.scatter(data.emissao,data.centralidade,s=32,color=CINZA,alpha=.78,edgecolors='white',linewidth=.5,zorder=2)
    for sid in rotulos:
        ax.scatter(data.loc[sid,'emissao'],data.loc[sid,'centralidade'],s=65,color=cores[sid],edgecolors='white',linewidth=.7,zorder=4)
    ax.grid(alpha=.15)
    ax.xaxis.set_major_formatter(pt)
    ax.yaxis.set_major_formatter(pt)
axes[0].set(title='A  Antes do ajuste',xlim=(-2,71),ylim=(-2,75),
    xlabel='Posto de emissão • 67 = maior',ylabel='Posto de centralidade • 67 = maior')
axes[0].set_xticks([1,20,40,67]); axes[0].set_yticks([1,20,40,67])
axes[1].axhline(0,color='#B9C3CA',lw=1,zorder=0)
axes[1].axvline(0,color='#B9C3CA',lw=1,zorder=0)
axes[1].set(title='B  Após ajuste pelo posto da produção',xlim=(-45,40),ylim=(-45,45),
    xlabel='Resíduo do posto de emissão',ylabel='Resíduo do posto de centralidade')
for ax,value,label in [(axes[0],rho,'Spearman'),(axes[1],parcial,'Parcial de postos')]:
    ax.text(.045,.965,label+' = '+f'{value:.3f}'.replace('.',','),transform=ax.transAxes,va='top',fontsize=12,
        bbox=dict(facecolor='white',edgecolor='#DFE5E8',pad=6))
handles = [Line2D([0],[0],marker='o',color='none',markerfacecolor=cores[sid],
    markeredgecolor='white',markersize=7,label=rotulos[sid]) for sid in rotulos]
fig.legend(handles=handles,loc='center',bbox_to_anchor=(.54,.13),ncol=3,frameon=False,fontsize=9)
acabamento(fig,4,'A associação se reduz após considerar a escala econômica',
    'Os mesmos 67 setores nos dois painéis • correlação bruta 0,383; parcial 0,158',
    'Fonte: outputs da investigação. Cenário ambiental exploratório. Ajuste linear dos postos por constante e posto de x.\nResíduo positivo: posto acima do previsto pelo ajuste. Não é um efeito causal nem uma porcentagem explicada.')
salvar(fig,'figura_4_ajuste_tamanho')
assert hashes == {f:hashlib.sha256((base/f).read_bytes()).hexdigest() for f in arquivos}
print('Entradas preservadas. Mediana da diferença absoluta dos postos:',np.median(abs(pd.Series(e).rank()-pd.Series(c).rank())))


Figura gravada: figura_4_ajuste_tamanho (PNG, SVG, PDF)
Entradas preservadas. Mediana da diferença absoluta dos postos: 16.0


![Associação bruta e parcial de postos](figura_4_ajuste_tamanho.png)

A estabilidade das matrizes não certifica o vetor ambiental. A auditoria e a classificação C estão documentadas em [auditoria dos coeficientes](../../../docs/auditoria_coeficientes_emissoes_2015.md).